# Notebook 5: LLM Inference Parameters
**LLM Fundamentals Demo Series — Agentic AI Bootcamp**

This notebook demonstrates how **Temperature**, **Top-K**, **Top-P**, and related inference parameters
directly control the token probability distribution that GPT-2 samples from.

We work directly with **raw logits** from GPT-2 so you can see the math happening live — not just
observe the output changing, but understand *why* it changes.

Topics covered:
1. The token probability distribution — logits and softmax
2. Temperature: sharpening and flattening the distribution
3. Greedy decoding vs. sampling
4. Top-K sampling
5. Top-P (Nucleus) sampling
6. Combining parameters: generation runs comparison
7. Frequency and Presence penalties (conceptual + API demo)
8. Stop sequences and max tokens

In [ ]:
# Install dependencies (run once)
!pip install transformers torch matplotlib seaborn numpy --quiet

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small, 117M params — downloads ~500 MB on first run)
MODEL_NAME  = 'gpt2'
tokenizer   = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model       = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.eval()

VOCAB_SIZE  = tokenizer.vocab_size
print(f'Model loaded: {MODEL_NAME}')
print(f'Vocabulary size: {VOCAB_SIZE:,}')
print(f'Max context length: {model.config.n_ctx} tokens')

## 1. The Token Probability Distribution

At each generation step, GPT-2 produces a **logit** (raw score) for every token in its vocabulary.
These are converted to probabilities via **softmax**:

$$P(w_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Let's inspect what the distribution looks like for a given prompt.

In [ ]:
def get_next_token_logits(prompt: str):
    """Run a forward pass and return the logits for the NEXT token."""
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model(input_ids)
    # Last token's logits → shape (vocab_size,)
    return output.logits[0, -1, :]


def top_k_probs(logits: torch.Tensor, k: int = 15, temperature: float = 1.0):
    """Return top-k (token_str, probability) pairs after applying temperature."""
    scaled = logits / temperature
    probs  = F.softmax(scaled, dim=-1)
    topk   = torch.topk(probs, k)
    tokens = [tokenizer.decode([idx.item()]) for idx in topk.indices]
    return list(zip(tokens, topk.values.numpy()))


# --- Run on a military-flavored prompt ---
PROMPT = "The mission objective was to secure the"
logits = get_next_token_logits(PROMPT)

pairs = top_k_probs(logits, k=15, temperature=1.0)
print(f'Prompt: "{PROMPT}"')
print(f'\nTop-15 next-token probabilities (Temperature = 1.0):')
print(f'{"Token":<20} {"Probability":>12}')
print('-' * 34)
for token, prob in pairs:
    bar = '█' * int(prob * 200)
    print(f'{repr(token):<20} {prob:>8.4f}  {bar}')

In [ ]:
# Visualize the full distribution (top-50 tokens)
pairs_50 = top_k_probs(logits, k=50, temperature=1.0)
tokens_50, probs_50 = zip(*pairs_50)

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(range(50), probs_50, color='steelblue', alpha=0.85, edgecolor='white')
ax.set_xticks(range(50))
ax.set_xticklabels([repr(t) for t in tokens_50], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('Probability')
ax.set_title(f'Next-Token Distribution: Top-50 Tokens\nPrompt: "{PROMPT}"', fontsize=12)
ax.set_xlabel('Token (ranked by probability)')
plt.tight_layout()
plt.show()
print(f'\nRemaining {VOCAB_SIZE - 50:,} tokens share: {1 - sum(probs_50):.4f} total probability')

## 2. Temperature: Sharpening and Flattening the Distribution

Temperature $T$ is applied **before** softmax by dividing the logits:

$$P(w_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- $T < 1$: Divides logits by a small number → **amplifies differences** → sharper/more deterministic
- $T = 1$: No change (identity)
- $T > 1$: Divides logits by a large number → **compresses differences** → flatter/more random

In [ ]:
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
K = 20  # top-K tokens to show

# Get top-20 tokens at T=1.0 to have a consistent x-axis
base_pairs = top_k_probs(logits, k=K, temperature=1.0)
base_tokens = [t for t, _ in base_pairs]
base_ids    = [tokenizer.encode(t, add_special_tokens=False)[0] for t in base_tokens]

fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4), sharey=False)

for ax, T in zip(axes, temperatures):
    scaled = logits / T
    probs  = F.softmax(scaled, dim=-1)
    selected_probs = probs[base_ids].numpy()
    
    color = plt.cm.RdYlGn(0.1 + 0.8 * (T / max(temperatures)))
    ax.bar(range(K), selected_probs, color=color, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(K))
    ax.set_xticklabels([repr(t) for t in base_tokens], rotation=70, ha='right', fontsize=6)
    ax.set_title(f'T = {T}', fontsize=12,
                 color='darkred' if T < 0.5 else ('darkgreen' if T > 1.2 else 'navy'))
    ax.set_ylabel('Probability' if ax == axes[0] else '')

plt.suptitle(f'Effect of Temperature on Next-Token Distribution\nPrompt: "{PROMPT}"', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Entropy quantifies distribution "spread"
from scipy.stats import entropy as scipy_entropy

print('Temperature vs. Distribution Entropy (higher = more random):')
print(f'{"Temperature":>15} {"Entropy (nats)":>16} {"Max Token Prob":>16}')
print('-' * 50)

for T in [0.1, 0.3, 0.5, 0.7, 1.0, 1.3, 1.7, 2.0]:
    scaled = logits / T
    probs  = F.softmax(scaled, dim=-1).numpy()
    ent    = scipy_entropy(probs)
    maxp   = probs.max()
    bar    = '░' * int(ent * 2)
    print(f'{T:>15.1f} {ent:>16.3f} {maxp:>16.4f}  {bar}')

## 3. Greedy Decoding vs. Sampling

**Greedy decoding** always picks the highest-probability token at each step (equivalent to T→0).
**Sampling** draws from the probability distribution, allowing lower-probability tokens to appear.

In [ ]:
def generate_text(prompt: str, max_new_tokens: int = 30,
                  do_sample: bool = False, temperature: float = 1.0,
                  top_k: int = 0, top_p: float = 1.0, seed: int = 42):
    """Generate text with configurable sampling strategy."""
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens   = max_new_tokens,
            do_sample        = do_sample,
            temperature      = temperature,
            top_k            = top_k if top_k > 0 else 0,
            top_p            = top_p,
            pad_token_id     = tokenizer.eos_token_id,
            repetition_penalty = 1.0,
        )
    # Decode only the newly generated tokens
    new_ids = output[0][len(input_ids[0]):]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


PROMPT_GEN = "The general ordered the troops to"
print(f'Prompt: "{PROMPT_GEN}"\n')

print('=== Greedy Decoding (deterministic) ===')
for _ in range(3):
    print('  >', generate_text(PROMPT_GEN, do_sample=False, seed=42))

print('\n=== Sampling (T=1.0) ===')
for seed in [1, 2, 3]:
    print('  >', generate_text(PROMPT_GEN, do_sample=True, temperature=1.0, seed=seed))

In [ ]:
# Show how temperature changes the text
print(f'Prompt: "{PROMPT_GEN}"\n')

for T in [0.1, 0.5, 1.0, 1.5, 2.0]:
    text = generate_text(PROMPT_GEN, do_sample=True, temperature=T, seed=7, max_new_tokens=25)
    label = 'deterministic' if T == 0.1 else ('creative' if T >= 1.5 else 'balanced')
    print(f'T={T:.1f} [{label:>13}]: {text}')

## 4. Top-K Sampling

**Top-K sampling** keeps only the $K$ highest-probability tokens and redistributes probability among them.
This prevents the model from selecting very improbable (incoherent) tokens.

$$\text{nucleus}_K = \{w \mid w \in \text{top-}K(P)\}$$

In [ ]:
def apply_top_k(probs: np.ndarray, k: int):
    """Zero out all but the top-K probabilities and renormalize."""
    result = np.zeros_like(probs)
    top_k_idx = np.argsort(probs)[-k:]
    result[top_k_idx] = probs[top_k_idx]
    result /= result.sum()
    return result


# Get baseline probabilities at T=1.0
base_probs = F.softmax(logits, dim=-1).numpy()
sorted_idx = np.argsort(base_probs)[::-1]  # descending
top50_probs = base_probs[sorted_idx[:50]]

k_values = [5, 20, 50, 200]
fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(22, 4))

# Baseline (no filter)
axes[0].bar(range(50), top50_probs, color='gray', alpha=0.7)
axes[0].set_title('Baseline (no Top-K)', fontsize=10)
axes[0].set_xlabel('Token rank')
axes[0].set_ylabel('Probability')

for ax, k in zip(axes[1:], k_values):
    filtered = apply_top_k(base_probs, k)
    top50_filtered = filtered[sorted_idx[:50]]
    
    colors = ['steelblue' if filtered[sorted_idx[i]] > 0 else 'lightgray' for i in range(50)]
    ax.bar(range(50), top50_filtered, color=colors, alpha=0.85)
    active = (filtered > 0).sum()
    ax.set_title(f'Top-K (K={k})\n{active} active tokens', fontsize=10)
    ax.set_xlabel('Token rank')

plt.suptitle(f'Top-K Sampling: Effect on Distribution\nPrompt: "{PROMPT}"', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Generate text with different Top-K values
print(f'Prompt: "{PROMPT_GEN}"\n')
print('Top-K sampling comparison (T=0.9, seed=42):')
print('-' * 70)

for k in [1, 5, 20, 50, 100]:
    text = generate_text(PROMPT_GEN, do_sample=True, temperature=0.9,
                         top_k=k, top_p=1.0, seed=42, max_new_tokens=25)
    print(f'  K={k:<5}: {text}')

## 5. Top-P (Nucleus) Sampling

**Top-P sampling** selects the smallest set of tokens whose cumulative probability exceeds $p$:

$$\text{nucleus}_p = \text{smallest } V' \subseteq V \text{ such that } \sum_{w \in V'} P(w) \geq p$$

Unlike Top-K, the nucleus **adapts to the model's confidence**:
- When confident → small nucleus (few tokens cover 95%)
- When uncertain → large nucleus (many tokens needed to cover 95%)

In [ ]:
def nucleus_size(probs: np.ndarray, p: float) -> int:
    """Return the number of tokens in the nucleus for a given p."""
    sorted_probs = np.sort(probs)[::-1]
    cumsum = np.cumsum(sorted_probs)
    return int(np.searchsorted(cumsum, p)) + 1


# Compare nucleus sizes across prompts with different certainty
prompts_test = [
    "The capital of France is",       # model is very confident
    "After careful analysis, the",    # more ambiguous
    "Some people believe that the",   # highly ambiguous
    "The soldier's primary weapon",   # moderate certainty
]

print(f'{"Prompt":<40} {"Nucleus size (p=0.90)":>22} {"Top token prob":>16}')
print('-' * 82)
for prompt in prompts_test:
    lgt   = get_next_token_logits(prompt)
    probs = F.softmax(lgt, dim=-1).numpy()
    n_size = nucleus_size(probs, 0.90)
    top_p  = probs.max()
    print(f'{prompt:<40} {n_size:>22} {top_p:>16.4f}')

In [ ]:
# Visualize the nucleus for different p values
lgt_demo = get_next_token_logits("The capital of France is")
probs_demo = F.softmax(lgt_demo, dim=-1).numpy()
sorted_probs_demo = np.sort(probs_demo)[::-1]
cumsum_demo = np.cumsum(sorted_probs_demo)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Cumulative probability curve
ax1.plot(range(1, 201), cumsum_demo[:200], color='steelblue', linewidth=2)
for p_val, color in [(0.90, 'red'), (0.95, 'orange'), (0.99, 'green')]:
    n = nucleus_size(probs_demo, p_val)
    ax1.axhline(p_val, linestyle='--', color=color, alpha=0.8, label=f'p={p_val} → nucleus={n}')
    ax1.axvline(n, linestyle='--', color=color, alpha=0.5)
ax1.set_xlabel('Number of tokens (ranked by prob)')
ax1.set_ylabel('Cumulative probability')
ax1.set_title('Nucleus Size for Confident Prompt\n"The capital of France is"')
ax1.legend(fontsize=9)
ax1.set_xlim(0, 200)

# Same for an uncertain prompt
lgt_unc = get_next_token_logits("Some people believe that the")
probs_unc = F.softmax(lgt_unc, dim=-1).numpy()
sorted_unc = np.sort(probs_unc)[::-1]
cumsum_unc = np.cumsum(sorted_unc)

ax2.plot(range(1, 201), cumsum_unc[:200], color='coral', linewidth=2)
for p_val, color in [(0.90, 'red'), (0.95, 'orange'), (0.99, 'green')]:
    n = nucleus_size(probs_unc, p_val)
    ax2.axhline(p_val, linestyle='--', color=color, alpha=0.8, label=f'p={p_val} → nucleus={n}')
    ax2.axvline(n, linestyle='--', color=color, alpha=0.5)
ax2.set_xlabel('Number of tokens (ranked by prob)')
ax2.set_title('Nucleus Size for Ambiguous Prompt\n"Some people believe that the"')
ax2.legend(fontsize=9)
ax2.set_xlim(0, 200)

plt.suptitle('Top-P Nucleus Adapts to Model Confidence', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Generate text with different Top-P values
print(f'Prompt: "{PROMPT_GEN}"\n')
print('Top-P (nucleus) sampling comparison (T=0.9, seed=42):')
print('-' * 70)

for p in [0.5, 0.75, 0.90, 0.95, 1.0]:
    text = generate_text(PROMPT_GEN, do_sample=True, temperature=0.9,
                         top_k=0, top_p=p, seed=42, max_new_tokens=25)
    print(f'  P={p:.2f}: {text}')

## 6. Full Parameter Comparison: Side-by-Side Runs

Let's run multiple generation strategies on the same prompt and compare output quality and diversity.

In [ ]:
PROMPT_COMPARE = "Intelligence reports indicate that hostile forces are"
N_RUNS = 4   # generate N times to show variance

configs = [
    {'label': 'Greedy (T=0, deterministic)',   'do_sample': False, 'temperature': 1.0, 'top_k': 0, 'top_p': 1.0},
    {'label': 'Low Temp (T=0.3)',              'do_sample': True,  'temperature': 0.3, 'top_k': 0, 'top_p': 1.0},
    {'label': 'Standard (T=0.7, p=0.9)',       'do_sample': True,  'temperature': 0.7, 'top_k': 0, 'top_p': 0.9},
    {'label': 'Creative (T=1.2, p=0.95)',      'do_sample': True,  'temperature': 1.2, 'top_k': 0, 'top_p': 0.95},
    {'label': 'High Temp (T=2.0)',             'do_sample': True,  'temperature': 2.0, 'top_k': 0, 'top_p': 1.0},
    {'label': 'Top-K=10 (T=0.8)',             'do_sample': True,  'temperature': 0.8, 'top_k': 10, 'top_p': 1.0},
]

print(f'Prompt: "{PROMPT_COMPARE}"')
print('=' * 80)

for cfg in configs:
    label = cfg['label']
    print(f'\n[ {label} ]')
    for seed in range(N_RUNS):
        text = generate_text(
            PROMPT_COMPARE,
            do_sample    = cfg['do_sample'],
            temperature  = cfg['temperature'],
            top_k        = cfg['top_k'],
            top_p        = cfg['top_p'],
            seed         = seed * 7 + 1,
            max_new_tokens = 30,
        )
        print(f'  Run {seed+1}: {text}')

## 7. Diversity Metric: Unique Output Ratio

We can quantify how much output varies by counting unique outputs across N runs.

In [ ]:
N_DIVERSITY_RUNS = 10

diversity_scores = []
for cfg in configs:
    outputs = []
    for seed in range(N_DIVERSITY_RUNS):
        text = generate_text(
            PROMPT_COMPARE,
            do_sample    = cfg['do_sample'],
            temperature  = cfg['temperature'],
            top_k        = cfg['top_k'],
            top_p        = cfg['top_p'],
            seed         = seed,
            max_new_tokens = 20,
        )
        outputs.append(text.strip())
    unique_ratio = len(set(outputs)) / N_DIVERSITY_RUNS
    diversity_scores.append(unique_ratio)
    print(f'{cfg["label"]:<40} Unique: {len(set(outputs)):>2}/{N_DIVERSITY_RUNS} ({unique_ratio:.0%})')

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
colors = plt.cm.RdYlGn([0.2, 0.35, 0.55, 0.7, 0.85, 0.6])
bars = ax.barh([cfg['label'] for cfg in configs], diversity_scores, color=colors, alpha=0.85)
ax.set_xlabel('Proportion of Unique Outputs (10 runs)')
ax.set_title('Output Diversity by Sampling Strategy', fontsize=12)
ax.set_xlim(0, 1.05)
for bar, score in zip(bars, diversity_scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.0%}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Repetition Penalties (Conceptual Demo)

GPT-2's `model.generate()` supports `repetition_penalty` (a multiplicative penalty on seen tokens).
The Anthropic/OpenAI APIs have separate `frequency_penalty` and `presence_penalty` parameters.

Here we demonstrate the repetition penalty built into HuggingFace.

In [ ]:
def generate_with_rep_penalty(prompt: str, rep_penalty: float = 1.0, max_new_tokens: int = 50):
    """Generate with repetition penalty (1.0 = none, >1.0 = penalize repeats)."""
    torch.manual_seed(42)
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens      = max_new_tokens,
            do_sample           = True,
            temperature         = 0.8,
            top_p               = 0.9,
            repetition_penalty  = rep_penalty,
            pad_token_id        = tokenizer.eos_token_id,
        )
    new_ids = output[0][len(input_ids[0]):]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


PROMPT_REP = "The operation required careful planning and careful"
print(f'Prompt: "{PROMPT_REP}"\n')

for rp in [1.0, 1.1, 1.3, 1.5, 2.0]:
    text = generate_with_rep_penalty(PROMPT_REP, rep_penalty=rp)
    label = 'no penalty' if rp == 1.0 else (f'penalty={rp}')
    print(f'  Rep={rp}: {text}')

## 9. Stop Sequences

Stop sequences halt generation when a specific string is produced.
In HuggingFace, we implement this with a custom `StoppingCriteria`. In commercial APIs (OpenAI, Anthropic),
it's a first-class parameter.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnSequence(StoppingCriteria):
    """Stop generation when any of the given token sequences is produced."""
    def __init__(self, stop_strings: list, tokenizer):
        self.stop_ids = [
            tokenizer.encode(s, add_special_tokens=False)
            for s in stop_strings
        ]

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        for stop_seq in self.stop_ids:
            if len(stop_seq) == 0:
                continue
            gen = input_ids[0].tolist()
            if gen[-len(stop_seq):] == stop_seq:
                return True
        return False


def generate_with_stop(prompt: str, stop_strings: list, max_new_tokens: int = 80):
    torch.manual_seed(42)
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    criteria  = StoppingCriteriaList([StopOnSequence(stop_strings, tokenizer)])
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens       = max_new_tokens,
            do_sample            = False,
            stopping_criteria    = criteria,
            pad_token_id         = tokenizer.eos_token_id,
        )
    new_ids = output[0][len(input_ids[0]):]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


PROMPT_STOP = "Mission summary: The unit advanced to the objective."
print(f'Prompt: "{PROMPT_STOP}"\n')

print('Without stop sequence (up to 80 tokens):')
print(' ', generate_with_stop(PROMPT_STOP, stop_strings=[], max_new_tokens=80))

print('\nWith stop sequence [".\\n", "END"]:')
print(' ', generate_with_stop(PROMPT_STOP, stop_strings=['.\n', 'END'], max_new_tokens=80))

## 10. Parameter Quick-Reference Summary

| Parameter | Range | Effect | Best For |
|-----------|-------|--------|----------|
| **Temperature** | 0.0–2.0 | Reshapes entire distribution | Primary creativity dial |
| **Top-K** | 1–vocab | Fixed candidate pool | Simple diversity control |
| **Top-P** | 0.0–1.0 | Adaptive nucleus | Preferred over Top-K |
| **Repetition Penalty** | 1.0–2.0 | Discourages repeats | Long-form text |
| **Stop Sequences** | List[str] | Halts at marker | Agent pipelines |
| **Max New Tokens** | 1–ctx | Output length cap | Cost + quality control |

### Recommended Configurations

```python
# Agent tool-calling / structured JSON
config_structured = dict(temperature=0.0, top_p=1.0, stop=["</tool_call>", "```"])

# Factual Q&A / summarization
config_factual = dict(temperature=0.2, top_p=0.9, max_new_tokens=300)

# Balanced conversational assistant
config_chat = dict(temperature=0.7, top_p=0.95, max_new_tokens=500)

# Creative writing / brainstorming
config_creative = dict(temperature=1.2, top_p=0.95, repetition_penalty=1.1, max_new_tokens=800)
```

> **Next notebook:** Prompt Engineering — controlling the model through input structure, not just sampling parameters.